# **Data Aquisition, Joining, and Cleaning**

> # Load all nine CSV files using pd.read_csv(). For files exceeding available RAM, use chunksize appropriately. Save the concatenated output as a Parquet file and report: (i) memory usage before and after downcasting, and (ii) file size of the resulting Parquet.

In [1]:
import numpy as np
import pandas as pd
import os
import glob

In [2]:
loans_master = pd.read_csv('data/raw/loans_master.csv')
loans_master.head()

,loan_id,issue_date,issue_year,issue_month,loan_amnt_inr,funded_amnt_inr,loan_term_months,int_rate_pct,installment_inr,annual_installment_inr,...,pymnt_plan,hardship_flag,initial_list_status,disbursement_method,verification_status,rbi_repo_rate_pct,gdp_growth_pct,cpi_inflation_pct,rate_spread_pct,real_interest_rate_pct
0,LN000000001,Feb-2016,2016,2,80678.0,74992.0,36,14.91,2793.17,33518.0,...,N,N,w,DIRECT_PAY,Source Verified,6.25,8.2,4.5,8.66,10.41
1,LN000000002,May-2024,2024,5,274166.0,265041.0,36,7.00,8465.45,101585.0,...,N,N,w,CASH,Verified,6.50,6.8,4.9,0.50,2.10
2,LN000000003,Dec-2021,2021,12,59603.0,54423.0,60,13.34,1366.55,16399.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,8.7,5.1,9.34,8.24
3,LN000000004,Nov-2020,2020,11,246313.0,224181.0,84,24.07,6088.88,73067.0,...,N,N,w,DIRECT_PAY,Source Verified,4.00,-6.6,6.2,20.07,17.87
4,LN000000005,Jul-2013,2013,7,101471.0,95361.0,60,8.52,2082.81,24994.0,...,Y,N,w,CASH,Verified,7.75,6.4,10.9,0.77,-2.38


In [3]:
loans_master.shape

(2000000, 27)

In [8]:
import os
import pandas as pd


def report_and_save_to_parquet(data_dict, output_folder="parquet_data"):
    os.makedirs(output_folder, exist_ok=True)

    for file_name, df in data_dict.items():
        parquet_path = os.path.join(output_folder, f"{file_name}.parquet")
        print(f"\n===== Processing: {file_name} =====")

        initial_mem_bytes = df.memory_usage(deep=True).sum()
        initial_mem_mb = initial_mem_bytes / (1024**2)
        print(f"Initial RAM Usage: {initial_mem_mb:.2f} MB")

        float_cols = df.select_dtypes(include=["float"]).columns
        for col in float_cols:
            df[col] = pd.to_numeric(df[col], downcast="float")

        int_cols = df.select_dtypes(include=["integer"]).columns
        for col in int_cols:
            df[col] = pd.to_numeric(df[col], downcast="integer")

        final_mem_bytes = df.memory_usage(deep=True).sum()
        final_mem_mb = final_mem_bytes / (1024**2)
        ram_savings = ((initial_mem_bytes - final_mem_bytes) / initial_mem_bytes) * 100
        print(f"Post-Downcast RAM Usage: {final_mem_mb:.2f} MB")
        print(f"RAM Savings: {ram_savings:.1f}%")

        try:
            df.to_parquet(parquet_path, engine="pyarrow", compression="snappy")

            file_size_bytes = os.path.getsize(parquet_path)
            file_size_mb = file_size_bytes / (1024**2)
            print(f"Final Parquet File Size: {file_size_mb:.2f} MB")

        except Exception as e:
            print(f"Error saving {file_name}: {e}")


report_and_save_to_parquet(data_dict)


In [9]:
# Join all nine tables on loan_id using sequential left merges. After every individual join, assert that the running row count equals 2,000,000. Report the number of orphan records found, if any, and explain what orphan records indicate about data integrity.
file_keys = [
    'loans_master',"credit_card_behavior", "collateral_assets", "loan_performance", "payment_history",
    "monthly_emi_track", "loan_enquiry_bureau", "branch_region_economy", "customer_bureau"
]

base_key = file_keys[0]
master_df = data_dict[base_key].copy()

assert(
    len(master_df) == 2000000
), f'Base table {base_key} does not have 2,000,000 rows! Current count: {len(master_df)}'

print(f' Initialized master table with {base_key} ({len(master_df):,} rows).')

# merging left the remaining 8 tbles
for next_key in file_keys[1:]:
  right_df = data_dict[next_key]

  # Calculating orphans before merging
  # They are the ID's present in the incoming table that do not exist in master loan list
  orphans_in_incoming = (~right_df['loan_id'].isin(master_df['loan_id'])).sum()
  if orphans_in_incoming > 0:
      print(f"Warning: Found {orphans_in_incoming:,} orphan records in {next_key}.\nOrphan records indicate a potential data integrity issue where child records exist without a corresponding parent record in the master table.")

  # Now we are perform the joining to the left
  master_df = pd.merge(master_df, right_df, on='loan_id', how='left')

  # asserting the running row count remains perfectly stable
  current_row_count = len(master_df)
  try:
    assert(
        current_row_count == 2000000
    )
  except AssertionError:
    print(f"Row count exploded to {current_row_count:,} after joining {next_key}.") # It indicates duplicate loan_id values in the right table when row count increase (Many-to-one explosion)

    raise
  print(f"Joined {next_key} successfully. Row count: {current_row_count:,} | Orphans bypassed: {orphans_in_incoming:,}")

print('\nAll merges completed successfully with a stable 2000000 row count.')

KeyError: 'loans_master'